In [1]:
from pathlib import Path
import json

import joblib
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

In [2]:
PROJECT_DIR = Path.cwd()
REPORTS_DIR = PROJECT_DIR / "reports"
MODELS_DIR = PROJECT_DIR / "models"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.20

print("Project directory:", PROJECT_DIR)
print("Reports directory:", REPORTS_DIR)
print("Models directory:", MODELS_DIR)

Project directory: c:\Users\Work\Desktop\Testable ML training and inference pipeline\Day2-Preprocessing and leakage-safe pipeline
Reports directory: c:\Users\Work\Desktop\Testable ML training and inference pipeline\Day2-Preprocessing and leakage-safe pipeline\reports
Models directory: c:\Users\Work\Desktop\Testable ML training and inference pipeline\Day2-Preprocessing and leakage-safe pipeline\models


In [3]:
dataset = load_breast_cancer(as_frame=True)

X = dataset.data.copy()
y = dataset.target.copy()

FEATURE_COLUMNS = X.columns.tolist()
TARGET_NAME = "target"

print("Dataset shape:", X.shape)
print("Number of features:", len(FEATURE_COLUMNS))
print("Target classes:", dict(enumerate(dataset.target_names)))

display(X.head())

Dataset shape: (569, 30)
Number of features: 30
Target classes: {0: np.str_('malignant'), 1: np.str_('benign')}


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

assert set(X_train.index).isdisjoint(set(X_test.index))

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Train-test overlap: PASSED")

Training rows: 455
Testing rows: 114
Train-test overlap: PASSED


In [5]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, FEATURE_COLUMNS),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

print("Preprocessing transformer created.")
print(preprocessor)

Preprocessing transformer created.
ColumnTransformer(transformers=[('numeric',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['mean radius', 'mean texture',
                                  'mean perimeter', 'mean area',
                                  'mean smoothness', 'mean compactness',
                                  'mean concavity', 'mean concave points',
                                  'mean symmetry', 'mean fractal dimension',
                                  'radius error', 'texture error',
                                  'perimeter error', 'area error',
                                  'smoothness error', 'compactness error',
                                  'concavity error', 'concave points error',
                                  'symmetry error', '

In [6]:
model_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=2000)),
    ]
)

print("Pipeline steps:")
for step_name, step in model_pipeline.steps:
    print(f"{step_name}: {type(step).__name__}")

Pipeline steps:
preprocessor: ColumnTransformer
classifier: LogisticRegression


In [7]:
model_pipeline.fit(X_train, y_train)

print("Pipeline training completed successfully.")
print("Preprocessor fitted:", hasattr(
    model_pipeline.named_steps["preprocessor"],
    "transformers_"
))
print("Classifier fitted:", hasattr(
    model_pipeline.named_steps["classifier"],
    "classes_"
))

Pipeline training completed successfully.
Preprocessor fitted: True
Classifier fitted: True


In [8]:
y_pred = model_pipeline.predict(X_test)

metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred, zero_division=0),
    "recall": recall_score(y_test, y_pred, zero_division=0),
    "f1_score": f1_score(y_test, y_pred, zero_division=0),
}

print("Test metrics:")
for metric_name, value in metrics.items():
    print(f"{metric_name}: {value:.4f}")

print("\nClassification report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=dataset.target_names,
        zero_division=0,
    )
)

print("Confusion matrix:")
display(pd.DataFrame(
    confusion_matrix(y_test, y_pred),
    index=["Actual malignant", "Actual benign"],
    columns=["Predicted malignant", "Predicted benign"],
))

pd.DataFrame([metrics]).to_csv(
    REPORTS_DIR / "day2_model_metrics.csv",
    index=False,
)

Test metrics:
accuracy: 0.9825
precision: 0.9861
recall: 0.9861
f1_score: 0.9861

Classification report:
              precision    recall  f1-score   support

   malignant       0.98      0.98      0.98        42
      benign       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114

Confusion matrix:


,Predicted malignant,Predicted benign
Actual malignant,41,1
Actual benign,1,71


In [9]:
def validate_input_schema(data):
    if not isinstance(data, pd.DataFrame):
        raise TypeError("Input must be a pandas DataFrame.")

    if data.empty:
        raise ValueError("Input contains no rows.")

    if not data.columns.is_unique:
        raise ValueError("Duplicate input column names found.")

    missing_columns = sorted(set(FEATURE_COLUMNS) - set(data.columns))
    extra_columns = sorted(set(data.columns) - set(FEATURE_COLUMNS))

    if missing_columns:
        raise ValueError(f"Missing required features: {missing_columns}")

    if extra_columns:
        raise ValueError(f"Unexpected extra features: {extra_columns}")

    # Enforce numerical input values.
    for column in FEATURE_COLUMNS:
        if not pd.api.types.is_numeric_dtype(data[column]):
            raise TypeError(f"Feature '{column}' must be numeric.")

    # Preserve the exact training feature order.
    return data.loc[:, FEATURE_COLUMNS].copy()


print("Schema validation function created.")

Schema validation function created.


In [10]:
valid_input = X_test.iloc[[0]].copy()

validated_input = validate_input_schema(valid_input)
prediction = model_pipeline.predict(validated_input)

print("Valid input test: PASSED")
print("Prediction:", int(prediction[0]))
print("Predicted diagnosis:", dataset.target_names[prediction[0]])

Valid input test: PASSED
Prediction: 0
Predicted diagnosis: malignant


In [11]:
single_row = X_test.iloc[[0]].copy()

single_row_prediction = model_pipeline.predict(
    validate_input_schema(single_row)
)

assert len(single_row_prediction) == 1

print("Single-row boundary test: PASSED")
print("Number of predictions:", len(single_row_prediction))

Single-row boundary test: PASSED
Number of predictions: 1


In [12]:
invalid_missing = X_test.iloc[[0]].drop(
    columns=[FEATURE_COLUMNS[0]]
)

try:
    validate_input_schema(invalid_missing)
except ValueError as error:
    print("Missing-feature negative test: PASSED")
    print("Expected error:", error)
else:
    raise AssertionError("Input with a missing feature was accepted.")

Missing-feature negative test: PASSED
Expected error: Missing required features: ['mean radius']


In [13]:
invalid_extra = X_test.iloc[[0]].copy()
invalid_extra["unexpected_feature"] = 123

try:
    validate_input_schema(invalid_extra)
except ValueError as error:
    print("Extra-feature negative test: PASSED")
    print("Expected error:", error)
else:
    raise AssertionError("Input with an extra feature was accepted.")

Extra-feature negative test: PASSED
Expected error: Unexpected extra features: ['unexpected_feature']


In [14]:
reordered_input = X_test.iloc[[0]][FEATURE_COLUMNS[::-1]]

validated_reordered = validate_input_schema(reordered_input)

assert validated_reordered.columns.tolist() == FEATURE_COLUMNS

original_prediction = model_pipeline.predict(
    validate_input_schema(X_test.iloc[[0]])
)
reordered_prediction = model_pipeline.predict(validated_reordered)

assert original_prediction[0] == reordered_prediction[0]

print("Reordered-column test: PASSED")
print("Schema validation restored the training feature order.")

Reordered-column test: PASSED
Schema validation restored the training feature order.


In [15]:
MODEL_PATH = MODELS_DIR / "day2_ml_pipeline.joblib"
SCHEMA_PATH = MODELS_DIR / "day2_schema.json"

joblib.dump(model_pipeline, MODEL_PATH)

schema = {
    "feature_columns": FEATURE_COLUMNS,
    "target_column": TARGET_NAME,
    "target_classes": {
        str(index): name
        for index, name in enumerate(dataset.target_names)
    },
    "model_type": "LogisticRegression",
    "preprocessing": [
        "SimpleImputer(strategy='median')",
        "StandardScaler()",
    ],
    "random_state": RANDOM_STATE,
}

with open(SCHEMA_PATH, "w", encoding="utf-8") as file:
    json.dump(schema, file, indent=4)

print("Pipeline saved:", MODEL_PATH)
print("Schema saved:", SCHEMA_PATH)

Pipeline saved: c:\Users\Work\Desktop\Testable ML training and inference pipeline\Day2-Preprocessing and leakage-safe pipeline\models\day2_ml_pipeline.joblib
Schema saved: c:\Users\Work\Desktop\Testable ML training and inference pipeline\Day2-Preprocessing and leakage-safe pipeline\models\day2_schema.json


In [16]:
loaded_pipeline = joblib.load(MODEL_PATH)

sample_input = validate_input_schema(X_test.iloc[[0]])

prediction_before_save = model_pipeline.predict(sample_input)
prediction_after_load = loaded_pipeline.predict(sample_input)

assert prediction_before_save[0] == prediction_after_load[0]

print("Model reload test: PASSED")
print("Prediction before saving:", int(prediction_before_save[0]))
print("Prediction after loading:", int(prediction_after_load[0]))

Model reload test: PASSED
Prediction before saving: 0
Prediction after loading: 0


In [17]:
test_results = [
    {"test": "Valid input", "status": "PASSED"},
    {"test": "Single-row input", "status": "PASSED"},
    {"test": "Missing feature rejected", "status": "PASSED"},
    {"test": "Extra feature rejected", "status": "PASSED"},
    {"test": "Reordered columns", "status": "PASSED"},
    {"test": "Reloaded model prediction", "status": "PASSED"},
]

test_report = pd.DataFrame(test_results)

display(test_report)

test_report.to_csv(
    REPORTS_DIR / "day2_inference_tests.csv",
    index=False,
)

print("Inference test evidence saved.")

,test,status
0,Valid input,PASSED
1,Single-row input,PASSED
2,Missing feature rejected,PASSED
3,Extra feature rejected,PASSED
4,Reordered columns,PASSED
5,Reloaded model prediction,PASSED


Inference test evidence saved.


In [19]:
assert len(X_train) == 455
assert len(X_test) == 114

assert set(X_train.index).isdisjoint(set(X_test.index))
assert list(model_pipeline.named_steps["preprocessor"].transformers_[0][2]) == FEATURE_COLUMNS

assert MODEL_PATH.exists()
assert SCHEMA_PATH.exists()

assert (test_report["status"] == "PASSED").all()

required_files = [
    REPORTS_DIR / "day2_model_metrics.csv",
    REPORTS_DIR / "day2_inference_tests.csv",
    PROJECT_DIR / "DAY2_PIPELINE_NOTES.md",
]

assert all(path.exists() for path in required_files)

print("=" * 55)
print("DAY 2 ACCEPTANCE CHECKS: PASSED")
print("=" * 55)
print("Preprocessing pipeline: PASSED")
print("Train-test separation: PASSED")
print("Inference schema validation: PASSED")
print("Negative and boundary tests: PASSED")
print("Model saving and reloading: PASSED")
print("Evidence and documentation: PASSED")
print("\nRemember to save your notebook with Ctrl + S.")

DAY 2 ACCEPTANCE CHECKS: PASSED
Preprocessing pipeline: PASSED
Train-test separation: PASSED
Inference schema validation: PASSED
Negative and boundary tests: PASSED
Model saving and reloading: PASSED
Evidence and documentation: PASSED

Remember to save your notebook with Ctrl + S.
